In [4]:
pip install -U nba_api

Note: you may need to restart the kernel to use updated packages.


In [1]:
"""
Close-game flag writer (saves progress continuously)

- Reads GAME_IDs from INPUT_FILE (must have column "GAME_ID")
- For each game, flags 1 if abs(margin at 5:00 in Q4) <= 5 else 0
- Writes each successful result immediately to OUT_CSV (append mode)
- Concurrent fetching with global rate limit + retry/backoff
- Live console progress + ETA
"""

import os
import re
import csv
import sys
import time
import math
import threading
from time import monotonic
from queue import Queue, Empty
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from nba_api.stats.endpoints import PlayByPlayV2
from requests.exceptions import ReadTimeout, RequestException

# ---------- CONFIG ----------
INPUT_FILE = "nba_highlights_with_pace_and_3pt_FIXED.csv"
OUT_CSV    = "game_close_with_5min.csv"

MAX_WORKERS     = 6       # concurrent games
RATE_LIMIT_RPS  = 2.5     # total requests/sec across threads (tune 1.5–3.5)
MAX_RETRIES     = 5       # retry attempts per game
RETRY_BACKOFF_S = 2.0     # backoff base (sleep = base * attempt)
REQUEST_TIMEOUT = 90.0    # seconds for each NBA API HTTP call

OUTPUT_COLS = ["GAME_ID", "close_game"]

# ---------- RATE LIMITER ----------
class TokenBucket:
    def __init__(self, rate_per_sec, capacity=None):
        self.rate = float(rate_per_sec)
        self.capacity = capacity or max(1.0, rate_per_sec * 2.0)
        self.tokens = self.capacity
        self.ts = time.monotonic()
        self.lock = threading.Lock()

    def consume(self, tokens=1.0):
        while True:
            with self.lock:
                now = time.monotonic()
                elapsed = now - self.ts
                self.ts = now
                self.tokens = min(self.capacity, self.tokens + elapsed * self.rate)
                if self.tokens >= tokens:
                    self.tokens -= tokens
                    return
            time.sleep(max(0.002, tokens / self.rate / 4))

BUCKET = TokenBucket(RATE_LIMIT_RPS)

# ---------- HELPERS ----------
def normalize_game_id(raw):
    s = re.sub(r"\D+", "", str(raw))
    return s.zfill(10)

def ensure_output_with_header():
    """Create output with header if missing."""
    if not os.path.exists(OUT_CSV):
        with open(OUT_CSV, "w", newline="") as f:
            csv.DictWriter(f, fieldnames=OUTPUT_COLS).writeheader()

def already_done_ids():
    """IDs already persisted to OUT_CSV (for resume)."""
    if not os.path.exists(OUT_CSV):
        return set()
    try:
        df = pd.read_csv(OUT_CSV, dtype=str, usecols=["GAME_ID"])
        return set(df["GAME_ID"].dropna().map(normalize_game_id))
    except Exception:
        return set()

def time_to_seconds_left(pctimestring):
    if not isinstance(pctimestring, str) or ":" not in pctimestring:
        return None
    mm, ss = pctimestring.split(":")
    try:
        return int(mm) * 60 + int(ss)
    except Exception:
        return None

def parse_margin(row_score, row_margin):
    if isinstance(row_margin, str):
        m = row_margin.strip().upper()
        if m == "TIE":
            return 0
        try:
            return int(row_margin)
        except Exception:
            pass
    if isinstance(row_score, str) and "-" in row_score:
        try:
            h, a = map(int, row_score.split("-"))
            return h - a
        except Exception:
            return None
    return None

def fmt_eta(seconds):
    if not seconds or seconds == float("inf"):
        return "ETA --:--:--"
    m, s = divmod(int(seconds), 60)
    h, m = divmod(m, 60)
    return f"ETA {h:02d}:{m:02d}:{s:02d}"

def rate_limited_pbp(game_id):
    """PlayByPlayV2 with global rate limit, retries, longer timeout."""
    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            BUCKET.consume(1.0)
            return PlayByPlayV2(game_id=game_id, timeout=REQUEST_TIMEOUT)
        except (ReadTimeout, RequestException, Exception) as e:
            last_err = e
            if attempt >= MAX_RETRIES:
                break
            time.sleep(RETRY_BACKOFF_S * attempt)
    raise last_err if last_err else RuntimeError("Unknown error in rate_limited_pbp")

def close_flag_from_pbp(game_id):
    """Return 1 if abs(margin) <= 5 at 5:00 in Q4, else 0."""
    pbp = rate_limited_pbp(game_id)
    df = pbp.get_data_frames()[0]
    if df.empty or "PERIOD" not in df.columns or "PCTIMESTRING" not in df.columns:
        raise ValueError("Invalid/empty PlayByPlay")

    q4 = df[df["PERIOD"] == 4].copy()
    if q4.empty:
        raise ValueError("No Q4 rows")

    q4["SEC_LEFT"] = q4["PCTIMESTRING"].map(time_to_seconds_left)
    q4 = q4[q4["SEC_LEFT"].notna()]

    pre5 = q4[q4["SEC_LEFT"] >= 300]
    row = pre5.iloc[-1] if not pre5.empty else q4.iloc[0]

    score = row["SCORE"] if "SCORE" in q4.columns else None
    margin = row["SCOREMARGIN"] if "SCOREMARGIN" in q4.columns else None
    val = parse_margin(score, margin)

    if val is None:
        prev = q4[q4["SCORE"].notna()]
        if not prev.empty:
            s = prev.iloc[-1]["SCORE"]
            val = parse_margin(s, None)

    if val is None:
        raise ValueError("Could not determine margin at 5:00 Q4")

    return 1 if abs(int(val)) <= 5 else 0

def process_one(gid):
    """Return dict row or None on failure (after retries)."""
    try:
        flag = close_flag_from_pbp(gid)
        return {"GAME_ID": gid, "close_game": flag}, None
    except Exception as e:
        return None, f"{gid}: {e}"

# ---------- WRITER THREAD ----------
def writer_worker(queue: Queue, stop_event: threading.Event):
    """Single writer that appends rows as they arrive (thread-safe)."""
    # Open once; write header only if file just created (done earlier)
    with open(OUT_CSV, "a", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=OUTPUT_COLS)
        while not stop_event.is_set() or not queue.empty():
            try:
                row = queue.get(timeout=0.25)
            except Empty:
                continue
            if row is None:
                # sentinel to flush/exit (optional)
                continue
            writer.writerow(row)
            # Ensure data is actually written (crash-safe)
            f.flush()

# ---------- MAIN ----------
def main():
    # Input GAME_IDs
    ids_df = pd.read_csv(INPUT_FILE, dtype=str, usecols=["GAME_ID"])
    game_ids = ids_df["GAME_ID"].dropna().map(normalize_game_id).unique().tolist()
    total = len(game_ids)
    print(f"Found {total} GAME_IDs in {INPUT_FILE}")

    # Output + resume
    ensure_output_with_header()
    done = already_done_ids()
    todo = [g for g in game_ids if g not in done]
    print(f"Resuming: {len(done)} already done, {len(todo)} remaining")
    if not todo:
        print("Nothing to do. Bye.")
        return

    # Start writer thread
    q = Queue(maxsize=2000)
    stop_event = threading.Event()
    wthr = threading.Thread(target=writer_worker, args=(q, stop_event), daemon=True)
    wthr.start()

    processed = succ = err = 0
    start = monotonic()
    last_report = start

    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(process_one, gid): gid for gid in todo}

        for fut in as_completed(futures):
            gid = futures[fut]
            row, error_msg = fut.result()
            processed += 1

            if row is not None:
                q.put(row)   # enqueue for immediate append
                succ += 1
            else:
                err += 1
                # Log once; skipped rows are not written
                print(f"\nERROR {error_msg}")

            # live progress every 25 completions or 3s
            now = monotonic()
            if processed % 25 == 0 or (now - last_report) >= 3.0:
                elapsed = now - start
                rps = processed / elapsed if elapsed > 0 else 0.0
                remaining = len(todo) - processed
                eta = remaining / rps if rps > 0 else float("inf")
                sys.stdout.write(
                    f"\rProcessed {processed}/{len(todo)} | "
                    f"Success: {succ}  Errors: {err} | "
                    f"Rate: {rps:.2f} req/s | {fmt_eta(eta)}"
                )
                sys.stdout.flush()
                last_report = now

    # Stop writer thread
    stop_event.set()
    wthr.join(timeout=5)

    print("\nDone.")
    print(f"Wrote {succ} rows to {OUT_CSV}. Errors: {err}")

if __name__ == "__main__":
    main()


Found 6731 GAME_IDs in nba_highlights_with_pace_and_3pt_FIXED.csv
Resuming: 4207 already done, 2524 remaining
Processed 1133/2524 | Success: 1133  Errors: 0 | Rate: 2.47 req/s | ETA 00:09:22

KeyboardInterrupt: 

In [1]:
"""
Pull %PTS_2PT_MR, %PTS_FB, %PTS_FT for GAME_IDs listed in ratings_final.csv
- Fixes KeyError 'resultSet' by normalizing GAME_IDs to 10-digit strings
- Robustly selects the 2-row team table from BoxScoreTraditionalV2
- Appends one row per game (resume-safe), prints progress
"""

import os
import time
import math
import re
import pandas as pd
from nba_api.stats.endpoints import BoxScoreSummaryV2, BoxScoreTraditionalV2, BoxScoreScoringV2

# ---------------- CONFIG ----------------
INPUT_FILE = "ratings_final.csv"
OUT_CSV = "game_pct_scoring_breakdown_from_ratings.csv"
SLEEP_BETWEEN_CALLS = 0.8
MAX_RETRIES = 5
RETRY_BACKOFF_SEC = 3

OUTPUT_COLS = [
    "GAME_ID",
    "GAME_DATE",
    "HOME_TEAM_ID",
    "HOME_TEAM_ABBR",
    "VISITOR_TEAM_ID",
    "VISITOR_TEAM_ABBR",
    "TOTAL_PTS",
    "PTS_2PT_MR_GAME",
    "PTS_FB_GAME",
    "PTS_FT_GAME",
    "ERROR_MSG"
]

# --------------- HELPERS ----------------
def normalize_game_id(raw):
    """Keep only digits and left-pad to 10 digits (NBA game IDs)."""
    s = re.sub(r"\D+", "", str(raw))
    if len(s) < 10:
        s = s.zfill(10)
    return s

def safe_call(fn, **kwargs):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fn(**kwargs)
        except Exception as e:
            if attempt < MAX_RETRIES:
                wait = RETRY_BACKOFF_SEC * attempt
                print(f"  -> Retry {attempt}/{MAX_RETRIES-1} after error: {e}. Waiting {wait}s...")
                time.sleep(wait)
            else:
                # re-raise on final attempt
                raise

def ensure_output():
    if not os.path.exists(OUT_CSV):
        pd.DataFrame(columns=OUTPUT_COLS).to_csv(OUT_CSV, index=False)

def already_done_ids():
    """Read done IDs and normalize them to 10 digits for robust resume."""
    if not os.path.exists(OUT_CSV):
        return set()
    try:
        df = pd.read_csv(OUT_CSV, dtype=str)
        if "GAME_ID" not in df.columns:
            return set()
        return set(df["GAME_ID"].dropna().map(normalize_game_id).tolist())
    except Exception:
        return set()

def append_row(row_dict):
    pd.DataFrame([row_dict], columns=OUTPUT_COLS).to_csv(
        OUT_CSV, mode="a", header=False, index=False
    )

def pull_summary_and_teams(game_id):
    """Get date, teams, and total points for a game."""
    summ = safe_call(BoxScoreSummaryV2, game_id=game_id)
    gs = summ.get_data_frames()[0]
    if gs.empty:
        raise ValueError("Empty GameSummary")
    game_date = str(gs.loc[0, "GAME_DATE_EST"])[:10]
    home_id = int(gs.loc[0, "HOME_TEAM_ID"])
    vis_id = int(gs.loc[0, "VISITOR_TEAM_ID"])

    trad = safe_call(BoxScoreTraditionalV2, game_id=game_id)
    frames = trad.get_data_frames()

    # Prefer the true 2-row team table
    team_df = None
    candidates = []
    for f in frames:
        cols = set(f.columns)
        if {"TEAM_ID", "PTS"}.issubset(cols):
            candidates.append(f.copy())

    # Choose the candidate with exactly 2 rows and TEAM_ABBREVIATION if possible
    for f in candidates:
        if len(f) == 2 and "TEAM_ABBREVIATION" in f.columns:
            team_df = f[["TEAM_ID", "TEAM_ABBREVIATION", "PTS"]].copy()
            break
    if team_df is None:
        # fallback: pick a candidate that has exactly 2 unique TEAM_IDs and aggregates to 2 rows
        for f in candidates:
            grp = f.groupby(["TEAM_ID", "TEAM_ABBREVIATION"], as_index=False)["PTS"].sum() if "TEAM_ABBREVIATION" in f.columns else f.groupby("TEAM_ID", as_index=False)["PTS"].sum()
            if len(grp) == 2:
                if "TEAM_ABBREVIATION" not in grp.columns:
                    grp["TEAM_ABBREVIATION"] = ""  # if missing, keep blank
                team_df = grp[["TEAM_ID", "TEAM_ABBREVIATION", "PTS"]].copy()
                break

    if team_df is None or team_df.empty:
        raise ValueError("Could not find 2-row team totals in BoxScoreTraditionalV2")

    team_df["TEAM_ID"] = pd.to_numeric(team_df["TEAM_ID"], errors="coerce").astype("Int64")
    team_df["PTS"] = pd.to_numeric(team_df["PTS"], errors="coerce").fillna(0).astype(int)
    total_pts = int(team_df["PTS"].sum())

    id_to_abbr = dict(zip(team_df["TEAM_ID"].astype(int), team_df["TEAM_ABBREVIATION"]))
    home_abbr = id_to_abbr.get(home_id, "")
    vis_abbr = id_to_abbr.get(vis_id, "")

    return game_date, home_id, vis_id, home_abbr, vis_abbr, total_pts, team_df[["TEAM_ID", "PTS"]]

def pull_team_pct_scoring(game_id):
    sc = safe_call(BoxScoreScoringV2, game_id=game_id)
    df = sc.get_data_frames()[0].copy()
    keep = ["TEAM_ID", "PCT_PTS_2PT_MR", "PCT_PTS_FB", "PCT_PTS_FT"]
    missing = [c for c in keep if c not in df.columns]
    if missing:
        raise ValueError(f"Missing columns in BoxScoreScoringV2: {missing}")
    for c in keep:
        if c != "TEAM_ID":
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df[keep]

def combine_game_percentages(team_pts_df, team_pct_df):
    merged = team_pts_df.merge(team_pct_df, on="TEAM_ID", how="inner")
    denom = merged["PTS"].sum()
    if merged.shape[0] < 2 or denom == 0:
        return {"PTS_2PT_MR_GAME": math.nan, "PTS_FB_GAME": math.nan, "PTS_FT_GAME": math.nan}

    def wadd(col): return (merged["PTS"] * merged[col]).sum() / denom
    return {
        "PTS_2PT_MR_GAME": wadd("PCT_PTS_2PT_MR"),
        "PTS_FB_GAME":     wadd("PCT_PTS_FB"),
        "PTS_FT_GAME":     wadd("PCT_PTS_FT"),
    }

# --------------- MAIN ----------------
def main():
    ids_df = pd.read_csv(INPUT_FILE, dtype=str)
    if "GAME_ID" not in ids_df.columns:
        raise ValueError("Input file must contain a column named GAME_ID.")

    # Normalize all input IDs to 10 digits
    game_ids = (
        ids_df["GAME_ID"]
        .dropna()
        .map(normalize_game_id)
        .unique()
        .tolist()
    )
    total = len(game_ids)
    print(f"Found {total} normalized GAME_IDs in {INPUT_FILE}. Output: {OUT_CSV}")

    ensure_output()
    done = already_done_ids()

    for i, gid in enumerate(game_ids, start=1):
        if gid in done:
            print(f"[{i}/{total}] {gid} already done. Skipping.")
            continue

        row = {
            "GAME_ID": gid, "GAME_DATE": "", "HOME_TEAM_ID": "",
            "HOME_TEAM_ABBR": "", "VISITOR_TEAM_ID": "",
            "VISITOR_TEAM_ABBR": "", "TOTAL_PTS": math.nan,
            "PTS_2PT_MR_GAME": math.nan, "PTS_FB_GAME": math.nan,
            "PTS_FT_GAME": math.nan, "ERROR_MSG": ""
        }

        try:
            game_date, home_id, vis_id, home_abbr, vis_abbr, total_pts, team_pts_df = pull_summary_and_teams(gid)
            time.sleep(SLEEP_BETWEEN_CALLS)
            team_pct_df = pull_team_pct_scoring(gid)
            time.sleep(SLEEP_BETWEEN_CALLS)
            combined = combine_game_percentages(team_pts_df, team_pct_df)

            row.update({
                "GAME_DATE": game_date,
                "HOME_TEAM_ID": home_id,
                "HOME_TEAM_ABBR": home_abbr,
                "VISITOR_TEAM_ID": vis_id,
                "VISITOR_TEAM_ABBR": vis_abbr,
                "TOTAL_PTS": int(total_pts),
                "PTS_2PT_MR_GAME": round(combined["PTS_2PT_MR_GAME"], 6) if pd.notna(combined["PTS_2PT_MR_GAME"]) else math.nan,
                "PTS_FB_GAME":     round(combined["PTS_FB_GAME"], 6)     if pd.notna(combined["PTS_FB_GAME"])     else math.nan,
                "PTS_FT_GAME":     round(combined["PTS_FT_GAME"], 6)     if pd.notna(combined["PTS_FT_GAME"])     else math.nan,
            })
        except Exception as e:
            row["ERROR_MSG"] = str(e)

        append_row(row)
        print(f"[{i}/{total}] {gid}  {row['VISITOR_TEAM_ABBR']}@{row['HOME_TEAM_ABBR']}  "
              f"PTS={row['TOTAL_PTS']}  2PT_MR%={row['PTS_2PT_MR_GAME']}  "
              f"FB%={row['PTS_FB_GAME']}  FT%={row['PTS_FT_GAME']}"
              + (f"  ERROR: {row['ERROR_MSG']}" if row['ERROR_MSG'] else ""))

    print(f"Done. Saved to {OUT_CSV}")

if __name__ == "__main__":
    main()


Found 195 normalized GAME_IDs in ratings_final.csv. Output: game_pct_scoring_breakdown_from_ratings.csv
[1/195] 0022300150 already done. Skipping.
[2/195] 0022300161 already done. Skipping.
[3/195] 0022300169 already done. Skipping.
[4/195] 0022300171 already done. Skipping.
[5/195] 0022300172 already done. Skipping.
[6/195] 0022300010 already done. Skipping.
[7/195] 0022300015 already done. Skipping.
[8/195] 0022300176 already done. Skipping.
[9/195] 0022300188 already done. Skipping.
[10/195] 0022300191 already done. Skipping.
[11/195] 0022300021 already done. Skipping.
[12/195] 0022300024 already done. Skipping.
[13/195] 0022300194 already done. Skipping.
[14/195] 0022300198 already done. Skipping.
[15/195] 0022300040 already done. Skipping.
[16/195] 0022300042 already done. Skipping.
[17/195] 0022300228 already done. Skipping.
[18/195] 0022300236 already done. Skipping.
[19/195] 0022300043 already done. Skipping.
[20/195] 0022300044 already done. Skipping.
[21/195] 0022300045 alrea

In [1]:
"""
Close-game flag at 5:00 remaining in Q4 for GAME_IDs in ratings_final.csv

Definition:
- "Close" if the absolute point differential at exactly 5:00 remaining in the 4th quarter
  is <= 5. (We take the most recent play at or above 5:00 remaining; i.e., the last event
  with PERIOD == 4 and SECONDS_REMAINING >= 300. If "SCOREMARGIN" is 'TIE', treat as 0.)

Behavior:
- Reads GAME_IDs from ratings_final.csv (column "GAME_ID"), normalizes to 10 digits.
- Appends one row per game to game_close_with_5min.csv, preserving progress.
- On errors, writes a row with ERROR_MSG but still does not skip the game.
- Prints progress lines like: [i/N] GAME_ID VIS@HOME ... close_game=1/0

Requires:
- nba_api
"""

import os
import re
import time
import math
import pandas as pd

from nba_api.stats.endpoints import BoxScoreSummaryV2, PlayByPlayV2
from requests.exceptions import RequestException

# ---------------- CONFIG ----------------
INPUT_FILE = "nba_highlights_with_pace_and_3pt_FIXED.csv"
OUT_CSV    = "game_close_with_5min.csv"

SLEEP_BETWEEN_CALLS = 0.4
MAX_RETRIES         = 5
RETRY_BACKOFF_SEC   = 3

OUTPUT_COLS = [
    "GAME_ID",
    "GAME_DATE",
    "HOME_TEAM_ID",
    "HOME_TEAM_ABBR",
    "VISITOR_TEAM_ID",
    "VISITOR_TEAM_ABBR",
    "SCORE_AT_5MIN_Q4",     # e.g., "101-98" (for debugging/validation)
    "MARGIN_AT_5MIN_Q4",    # integer margin from the play-by-play perspective ("SCOREMARGIN")
    "close_game",           # 1 if abs(margin) <= 5 at 5:00 Q4, else 0
    "ERROR_MSG"
]

# ---------------- HELPERS ----------------
def normalize_game_id(raw):
    """Keep only digits and left-pad to 10 digits (e.g., 22300150 -> 0022300150)."""
    s = re.sub(r"\D+", "", str(raw))
    if len(s) < 10:
        s = s.zfill(10)
    return s

def safe_call(fn, **kwargs):
    """Call an nba_api endpoint with retries + backoff."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return fn(**kwargs)
        except Exception as e:
            if attempt < MAX_RETRIES:
                wait = RETRY_BACKOFF_SEC * attempt
                print(f"  -> Retry {attempt}/{MAX_RETRIES-1} after error: {e}. Waiting {wait}s...")
                time.sleep(wait)
            else:
                raise

def ensure_output():
    """Create the output CSV with header if missing."""
    if not os.path.exists(OUT_CSV):
        pd.DataFrame(columns=OUTPUT_COLS).to_csv(OUT_CSV, index=False)

def already_done_ids():
    """Return normalized GAME_IDs already written to OUT_CSV (for resume)."""
    if not os.path.exists(OUT_CSV):
        return set()
    try:
        df = pd.read_csv(OUT_CSV, dtype=str)
        if "GAME_ID" not in df.columns:
            return set()
        return set(df["GAME_ID"].dropna().map(normalize_game_id).tolist())
    except Exception:
        return set()

def append_row(row_dict):
    """Append one row to OUT_CSV without rewriting the whole file."""
    pd.DataFrame([row_dict], columns=OUTPUT_COLS).to_csv(
        OUT_CSV, mode="a", header=False, index=False
    )

def pull_summary_and_teams(game_id):
    """
    From BoxScoreSummaryV2:
      - GAME_DATE_EST, HOME_TEAM_ID, VISITOR_TEAM_ID
      - We also want team abbreviations. Those aren't always in Summary, so we
        infer via PlayByPlay (team abbreviations not guaranteed there either).
      - Easiest: use the 'HOME_TEAM_ABBREVIATION'/'VISITOR_TEAM_ABBREVIATION'
        if present in summary (some versions include), else leave blank strings.
    """
    summ = safe_call(BoxScoreSummaryV2, game_id=game_id)
    gs = summ.get_data_frames()[0]
    if gs.empty:
        raise ValueError("Empty GameSummary")

    game_date = str(gs.loc[0, "GAME_DATE_EST"])[:10] if "GAME_DATE_EST" in gs.columns else ""
    home_id   = int(gs.loc[0, "HOME_TEAM_ID"])
    vis_id    = int(gs.loc[0, "VISITOR_TEAM_ID"])

    # Try to get abbreviations if provided by this nba_api version
    home_abbr = ""
    vis_abbr  = ""
    for cand in ["HOME_TEAM_ABBREVIATION", "HOME_TEAM_ABBR"]:
        if cand in gs.columns:
            home_abbr = str(gs.loc[0, cand])
            break
    for cand in ["VISITOR_TEAM_ABBREVIATION", "VISITOR_TEAM_ABBR"]:
        if cand in gs.columns:
            vis_abbr = str(gs.loc[0, cand])
            break

    return game_date, home_id, vis_id, home_abbr, vis_abbr

def time_to_seconds_left(pctimestring):
    """
    Convert "MM:SS" (PCTIMESTRING) to seconds remaining in period.
    """
    if not isinstance(pctimestring, str) or ":" not in pctimestring:
        return None
    mm, ss = pctimestring.split(":")
    try:
        return int(mm) * 60 + int(ss)
    except Exception:
        return None

def get_margin_at_5min_q4(game_id):
    """
    Using PlayByPlayV2, find the last row in Q4 with SECONDS_REMAINING >= 300,
    then return (score_string, margin_int).
      - If SCREMARGIN is 'TIE', treat margin as 0.
      - If SCOREMARGIN is missing, parse SCORE like '101-98'.
    """
    pbp = safe_call(PlayByPlayV2, game_id=game_id)
    df = pbp.get_data_frames()[0].copy()
    if df.empty:
        raise ValueError("Empty PlayByPlay data")

    # Ensure required columns exist
    for col in ["PERIOD", "PCTIMESTRING"]:
        if col not in df.columns:
            raise ValueError(f"PlayByPlay missing column: {col}")

    # Compute seconds remaining in period
    df["SEC_LEFT"] = df["PCTIMESTRING"].map(time_to_seconds_left)

    # Filter to Q4 rows that occur at/after 5:00 (>= 300 seconds remaining)
    q4 = df[(df["PERIOD"] == 4) & (df["SEC_LEFT"].notna()) & (df["SEC_LEFT"] >= 300)]
    if q4.empty:
        # If no such rows (should be rare), take the earliest Q4 row as fallback
        q4 = df[df["PERIOD"] == 4]
        if q4.empty:
            raise ValueError("No 4th quarter play-by-play rows found.")
        # Use the first Q4 row as the state closest to 12:00; margin might be NaN
        row = q4.iloc[0]
    else:
        # We want the last event BEFORE the clock dips below 5:00,
        # i.e., the last with SEC_LEFT >= 300
        row = q4.iloc[-1]

    score_str   = row["SCORE"] if "SCORE" in row and isinstance(row["SCORE"], str) else None
    margin_raw  = row["SCOREMARGIN"] if "SCOREMARGIN" in row else None

    # Interpret margin
    margin_val = None
    if isinstance(margin_raw, str):
        if margin_raw.strip().upper() == "TIE":
            margin_val = 0
        else:
            try:
                margin_val = int(margin_raw)
            except Exception:
                margin_val = None

    # If margin missing, parse from SCORE "home-away"
    if margin_val is None:
        if isinstance(score_str, str) and "-" in score_str:
            try:
                home, away = score_str.split("-")
                home = int(home)
                away = int(away)
                margin_val = home - away  # positive means home leading
            except Exception:
                margin_val = None

    # It’s possible both are None if SCORE isn’t set on that row;
    # in that case, search backward in q4 for the most recent non-null SCORE.
    if margin_val is None:
        # search earlier within Q4 subset for a SCORE value
        prev = q4[q4["SCORE"].notna()]
        if not prev.empty:
            last_score = prev.iloc[-1]["SCORE"]
            if isinstance(last_score, str) and "-" in last_score:
                try:
                    home, away = map(int, last_score.split("-"))
                    margin_val = home - away
                    score_str  = last_score
                except Exception:
                    pass

    return score_str, margin_val

# ---------------- MAIN ----------------
def main():
    ids_df = pd.read_csv(INPUT_FILE, dtype=str)
    if "GAME_ID" not in ids_df.columns:
        raise ValueError("Input file must contain a column named GAME_ID.")

    game_ids = (
        ids_df["GAME_ID"]
        .dropna()
        .map(normalize_game_id)
        .unique()
        .tolist()
    )
    total = len(game_ids)
    print(f"Found {total} normalized GAME_IDs in {INPUT_FILE}. Output: {OUT_CSV}")

    ensure_output()
    done = already_done_ids()

    for i, gid in enumerate(game_ids, start=1):
        if gid in done:
            print(f"[{i}/{total}] {gid} already done. Skipping.")
            continue

        row = {
            "GAME_ID": gid,
            "GAME_DATE": "",
            "HOME_TEAM_ID": "",
            "HOME_TEAM_ABBR": "",
            "VISITOR_TEAM_ID": "",
            "VISITOR_TEAM_ABBR": "",
            "SCORE_AT_5MIN_Q4": "",
            "MARGIN_AT_5MIN_Q4": "",
            "close_game": math.nan,
            "ERROR_MSG": ""
        }

        try:
            # Date + teams (IDs & possible abbreviations)
            game_date, home_id, vis_id, home_abbr, vis_abbr = pull_summary_and_teams(gid)
            row.update({
                "GAME_DATE": game_date,
                "HOME_TEAM_ID": home_id,
                "HOME_TEAM_ABBR": home_abbr,
                "VISITOR_TEAM_ID": vis_id,
                "VISITOR_TEAM_ABBR": vis_abbr,
            })
            time.sleep(SLEEP_BETWEEN_CALLS)

            # Margin at 5:00 in Q4
            score_str, margin_val = get_margin_at_5min_q4(gid)
            row["SCORE_AT_5MIN_Q4"]  = score_str if isinstance(score_str, str) else ""
            row["MARGIN_AT_5MIN_Q4"] = None if margin_val is None else int(margin_val)

            if margin_val is None:
                raise ValueError("Could not determine margin at 5:00 in Q4 from play-by-play.")

            row["close_game"] = 1 if abs(int(margin_val)) <= 5 else 0

        except Exception as e:
            row["ERROR_MSG"] = str(e)

        append_row(row)

        # Pretty console line
        vis = row["VISITOR_TEAM_ABBR"] or "VIS"
        home = row["HOME_TEAM_ABBR"] or "HOME"
        print(
            f"[{i}/{total}] {gid} {vis}@{home} "
            f"5:00Q4 score={row['SCORE_AT_5MIN_Q4']} margin={row['MARGIN_AT_5MIN_Q4']} "
            f"close_game={row['close_game']}"
            + (f"  ERROR: {row['ERROR_MSG']}" if row['ERROR_MSG'] else "")
        )

        time.sleep(SLEEP_BETWEEN_CALLS)

    print(f"Done. Saved to {OUT_CSV}")

if __name__ == "__main__":
    main()


Found 6731 normalized GAME_IDs in nba_highlights_with_pace_and_3pt_FIXED.csv. Output: game_close_with_5min.csv
[1/6731] 0021900001 already done. Skipping.
[2/6731] 0021900002 already done. Skipping.
[3/6731] 0021900003 already done. Skipping.
[4/6731] 0021900005 already done. Skipping.
[5/6731] 0021900004 already done. Skipping.
[6/6731] 0021900007 already done. Skipping.
[7/6731] 0021900006 already done. Skipping.
[8/6731] 0021900008 already done. Skipping.
[9/6731] 0021900010 already done. Skipping.
[10/6731] 0021900009 already done. Skipping.
[11/6731] 0021900011 already done. Skipping.
[12/6731] 0021900012 already done. Skipping.
[13/6731] 0021900013 already done. Skipping.
[14/6731] 0021900014 already done. Skipping.
[15/6731] 0021900015 already done. Skipping.
[16/6731] 0021900016 already done. Skipping.
[17/6731] 0021900018 already done. Skipping.
[18/6731] 0021900017 already done. Skipping.
[19/6731] 0021900022 already done. Skipping.
[20/6731] 0021900019 already done. Skipping

KeyboardInterrupt: 

In [ ]:
## OFF Rating?

In [ ]:
## TO %

In [7]:
## Player Participation by Game (Played / DNP / Inactive) — rostered-only
# --- Packages -----------------------------------------------------------------
import os
import time
from pathlib import Path
import pandas as pd

from nba_api.stats.endpoints import LeagueGameLog, BoxScoreTraditionalV2, BoxScoreSummaryV2

# --- Config -------------------------------------------------------------------
SEASON = "2023-24"             # e.g., "2024-25"
SEASON_TYPE = "Regular Season" # "Regular Season", "Playoffs", etc.
SLEEP_BETWEEN_CALLS = 1.2      # seconds between requests
MAX_RETRIES = 4

# Input: CSV you provide with at least columns PERSON_ID (int) and PLAYER_NAME (optional, used for nicer output)
PLAYERS_CSV = Path("all_nba_2024_25_filtered_min.csv")

# Output: incremental CSV; one row per (game x player)
OUTFILE = Path(f"player_participation_incremental_{SEASON.replace('-', '')}.csv")  # -> player_participation_incremental_201516.csv

# --- Helpers ------------------------------------------------------------------
def load_players_list(csv_path: Path) -> pd.DataFrame:
    """Load the universe of players to track. Expect PERSON_ID (int). Optional: PLAYER_NAME."""
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing players CSV: {csv_path.resolve()}")
    df = pd.read_csv(csv_path)
    if "PERSON_ID" not in df.columns:
        raise ValueError("players_master.csv must have a PERSON_ID column (NBA player id).")
    # standardize types/columns
    df["PERSON_ID"] = pd.to_numeric(df["PERSON_ID"], errors="coerce").astype("Int64")
    if "PLAYER_NAME" not in df.columns:
        df["PLAYER_NAME"] = pd.NA
    return df.dropna(subset=["PERSON_ID"]).copy()

def load_done_ids(outfile: Path) -> set:
    """Return set of GAME_IDs already written to the CSV (so we don't reprocess)."""
    if not outfile.exists():
        return set()
    try:
        done = pd.read_csv(outfile, usecols=["GAME_ID"], dtype=str)
        return set(done["GAME_ID"].astype(str).unique())
    except Exception:
        return set()

def ensure_header(outfile: Path):
    """Create file with header if it doesn't exist."""
    if outfile.exists():
        return
    cols = [
        "GAME_ID", "GAME_DATE", "HOME_TEAM", "AWAY_TEAM",
        "TEAM_ABBR", "PLAYER_ID", "PLAYER_NAME", "STATUS"
    ]
    pd.DataFrame(columns=cols).to_csv(outfile, index=False)

def fetch_team_gamelog(season: str, season_type: str) -> pd.DataFrame:
    gl = LeagueGameLog(
        season=season,
        season_type_all_star=season_type,
        player_or_team_abbreviation="T",
        timeout=60
    )
    df = gl.get_data_frames()[0].copy()
    if "GAME_DATE" in df.columns:
        df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
    return df[["GAME_ID", "GAME_DATE", "TEAM_ABBREVIATION", "MATCHUP"]].copy()

def fetch_boxscore_summary(game_id: str, max_retries=MAX_RETRIES, sleep=SLEEP_BETWEEN_CALLS):
    """Return (df_available, df_inactive) from BoxScoreSummaryV2 for a game."""
    for attempt in range(max_retries):
        try:
            bs = BoxScoreSummaryV2(game_id=game_id, timeout=60)
            df_avail = bs.available_players.get_data_frame().copy()
            df_inac  = bs.inactive_players.get_data_frame().copy()
            return df_avail, df_inac
        except Exception:
            time.sleep(sleep * (2 ** attempt))
    return pd.DataFrame(), pd.DataFrame()

def fetch_player_traditional(game_id: str, max_retries=MAX_RETRIES, sleep=SLEEP_BETWEEN_CALLS) -> pd.DataFrame:
    """Player traditional box score (rows: players who recorded a statline)."""
    for attempt in range(max_retries):
        try:
            bs = BoxScoreTraditionalV2(game_id=game_id, timeout=60)
            return bs.player_stats.get_data_frame().copy()
        except Exception:
            time.sleep(sleep * (2 ** attempt))
    return pd.DataFrame()

def derive_home_away(gdf: pd.DataFrame):
    """Given the two team rows for a game from LeagueGameLog, return (home, away, date)."""
    game_date = gdf["GAME_DATE"].iloc[0] if "GAME_DATE" in gdf.columns else None
    home_row = gdf[gdf["MATCHUP"].str.contains("vs.", na=False)]
    away_row = gdf[gdf["MATCHUP"].str.contains("@",   na=False)]
    home = home_row["TEAM_ABBREVIATION"].iloc[0] if not home_row.empty else None
    away = away_row["TEAM_ABBREVIATION"].iloc[0] if not away_row.empty else None
    return home, away, game_date

def write_rows(outfile: Path, rows: list[dict]):
    """Append multiple rows to CSV immediately."""
    if not rows:
        return
    pd.DataFrame(rows).to_csv(outfile, mode="a", index=False, header=False)

# --- Main ---------------------------------------------------------------------
def main():
    # Load universe of players to track
    players_master = load_players_list(PLAYERS_CSV)
    tracked_ids = set(players_master["PERSON_ID"].astype(int).tolist())
    name_lut = dict(zip(players_master["PERSON_ID"].astype(int), players_master["PLAYER_NAME"]))

    ensure_header(OUTFILE)
    done_ids = load_done_ids(OUTFILE)

    tlog = fetch_team_gamelog(SEASON, SEASON_TYPE)
    if tlog.empty:
        raise RuntimeError("LeagueGameLog returned no rows. Check SEASON / SEASON_TYPE.")

    groups = list(tlog.groupby("GAME_ID"))
    groups.sort(key=lambda kv: kv[1]["GAME_DATE"].iloc[0])

    total = len(groups)
    wrote_games = 0
    skipped = 0

    print(f"Found {total} game_ids for {SEASON} ({SEASON_TYPE}). Output: {OUTFILE}")

    for idx, (gid, gdf) in enumerate(groups, start=1):
        gid = str(gid)
        if gid in done_ids:
            skipped += 1
            if idx % 25 == 0 or idx == total:
                print(f"[{idx}/{total}] Skipped already done GAME_ID {gid} (skipped={skipped})")
            continue

        home, away, game_date = derive_home_away(gdf)

        # Pull participation ingredients
        df_avail, df_inac = fetch_boxscore_summary(gid)
        df_play = fetch_player_traditional(gid)
        time.sleep(SLEEP_BETWEEN_CALLS)

        # Build sets
        avail_ids = set(pd.to_numeric(df_avail.get("PERSON_ID", pd.Series(dtype="Int64")), errors="coerce").dropna().astype(int).tolist()) if not df_avail.empty else set()
        inact_ids = set(pd.to_numeric(df_inac.get("PERSON_ID", pd.Series(dtype="Int64")),  errors="coerce").dropna().astype(int).tolist()) if not df_inac.empty else set()
        played_ids = set(pd.to_numeric(df_play.get("PLAYER_ID", pd.Series(dtype="Int64")), errors="coerce").dropna().astype(int).tolist()) if not df_play.empty else set()

        # --- KEY CHANGE: only track players actually on this game's rosters ---
        rostered_ids = (avail_ids | inact_ids | played_ids) & tracked_ids

        # Team abbreviations per rostered player
        team_cols = ["PERSON_ID", "TEAM_ABBREVIATION"]
        team_map_df = pd.concat([
            df_avail[team_cols] if not df_avail.empty and all(c in df_avail.columns for c in team_cols) else pd.DataFrame(columns=team_cols),
            df_inac[team_cols]  if not df_inac.empty  and all(c in df_inac.columns  for c in team_cols) else pd.DataFrame(columns=team_cols),
        ], axis=0, ignore_index=True).dropna(subset=["PERSON_ID"]).drop_duplicates("PERSON_ID")
        team_map = {int(r.PERSON_ID): r.TEAM_ABBREVIATION for _, r in team_map_df.iterrows()}

        # Name lookup fallback from APIs if your CSV is missing some names
        name_map = name_lut.copy()
        for src_df, id_col, name_col in [
            (df_avail, "PERSON_ID", "PLAYER_NAME"),
            (df_inac, "PERSON_ID", "PLAYER_NAME"),
            (df_play.rename(columns={"PLAYER_ID":"PERSON_ID"}), "PERSON_ID", "PLAYER_NAME"),
        ]:
            if not src_df.empty and all(c in src_df.columns for c in [id_col, name_col]):
                for _, r in src_df.iterrows():
                    pid = pd.to_numeric(r[id_col], errors="coerce")
                    if pd.notna(pid):
                        pid = int(pid)
                        if pid in rostered_ids and not name_map.get(pid):
                            name_map[pid] = r.get(name_col, None)

        # Build rows ONLY for rostered tracked players
        rows = []
        for pid in sorted(rostered_ids):
            if pid in inact_ids:
                status = "Inactive"
            elif pid in played_ids:
                status = "Played"
            elif pid in avail_ids:
                status = "DNP"
            else:
                # shouldn't happen because we restricted to rostered_ids, but keep guard
                continue

            rows.append({
                "GAME_ID": gid,
                "GAME_DATE": pd.to_datetime(game_date).date() if pd.notna(game_date) else None,
                "HOME_TEAM": home,
                "AWAY_TEAM": away,
                "TEAM_ABBR": team_map.get(pid),
                "PLAYER_ID": pid,
                "PLAYER_NAME": name_map.get(pid),
                "STATUS": status
            })

        # If neither team had any tracked players, write nothing for this game
        if rows:
            write_rows(OUTFILE, rows)

        done_ids.add(gid)
        wrote_games += 1

        # Live summary (only rostered tracked players)
        played_ct   = sum(1 for r in rows if r["STATUS"] == "Played")
        dnp_ct      = sum(1 for r in rows if r["STATUS"] == "DNP")
        inactive_ct = sum(1 for r in rows if r["STATUS"] == "Inactive")
        print(
            f"[{idx}/{total}] {gid}  {away} @ {home}  date={pd.to_datetime(game_date).date() if pd.notna(game_date) else None}  "
            f"TrackedRostered={len(rows)}  Played={played_ct}  DNP={dnp_ct}  Inactive={inactive_ct}"
        )

    print(f"Done. Processed {wrote_games} games. Skipped {skipped} already present.")

if __name__ == "__main__":
    main()


Found 1230 game_ids for 2024-25 (Regular Season). Output: player_participation_incremental_202425.csv
[4/1230] 0022400064  BKN @ ATL  date=2024-10-23  TrackedRostered=0  Played=0  DNP=0  Inactive=0


KeyboardInterrupt: 

In [7]:
## Player Participation by Game (Played / DNP / Inactive) — rostered-only
# --- Packages -----------------------------------------------------------------
import os
import time
from pathlib import Path
import pandas as pd

from nba_api.stats.endpoints import LeagueGameLog, BoxScoreTraditionalV2, BoxScoreSummaryV2

# --- Config -------------------------------------------------------------------
SEASON = "2023-24"             # e.g., "2024-25"
SEASON_TYPE = "Regular Season" # "Regular Season", "Playoffs", etc.
SLEEP_BETWEEN_CALLS = 1.2      # seconds between requests
MAX_RETRIES = 4

# Input: CSV you provide with at least columns PERSON_ID (int) and PLAYER_NAME (optional, used for nicer output)
PLAYERS_CSV = Path("all_nba_2024_25_filtered_min.csv")

# Output: incremental CSV; one row per (game x player)
OUTFILE = Path(f"player_participation_incremental_{SEASON.replace('-', '')}.csv")  # -> player_participation_incremental_201516.csv

# --- Helpers ------------------------------------------------------------------
def load_players_list(csv_path: Path) -> pd.DataFrame:
    """Load the universe of players to track. Expect PERSON_ID (int). Optional: PLAYER_NAME."""
    if not csv_path.exists():
        raise FileNotFoundError(f"Missing players CSV: {csv_path.resolve()}")
    df = pd.read_csv(csv_path)
    if "PERSON_ID" not in df.columns:
        raise ValueError("players_master.csv must have a PERSON_ID column (NBA player id).")
    # standardize types/columns
    df["PERSON_ID"] = pd.to_numeric(df["PERSON_ID"], errors="coerce").astype("Int64")
    if "PLAYER_NAME" not in df.columns:
        df["PLAYER_NAME"] = pd.NA
    return df.dropna(subset=["PERSON_ID"]).copy()

def load_done_ids(outfile: Path) -> set:
    """Return set of GAME_IDs already written to the CSV (so we don't reprocess)."""
    if not outfile.exists():
        return set()
    try:
        done = pd.read_csv(outfile, usecols=["GAME_ID"], dtype=str)
        return set(done["GAME_ID"].astype(str).unique())
    except Exception:
        return set()

def ensure_header(outfile: Path):
    """Create file with header if it doesn't exist."""
    if outfile.exists():
        return
    cols = [
        "GAME_ID", "GAME_DATE", "HOME_TEAM", "AWAY_TEAM",
        "TEAM_ABBR", "PLAYER_ID", "PLAYER_NAME", "STATUS"
    ]
    pd.DataFrame(columns=cols).to_csv(outfile, index=False)

def fetch_team_gamelog(season: str, season_type: str) -> pd.DataFrame:
    gl = LeagueGameLog(
        season=season,
        season_type_all_star=season_type,
        player_or_team_abbreviation="T",
        timeout=60
    )
    df = gl.get_data_frames()[0].copy()
    if "GAME_DATE" in df.columns:
        df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
    return df[["GAME_ID", "GAME_DATE", "TEAM_ABBREVIATION", "MATCHUP"]].copy()

def fetch_boxscore_summary(game_id: str, max_retries=MAX_RETRIES, sleep=SLEEP_BETWEEN_CALLS):
    """Return (df_available, df_inactive) from BoxScoreSummaryV2 for a game."""
    for attempt in range(max_retries):
        try:
            bs = BoxScoreSummaryV2(game_id=game_id, timeout=60)
            df_avail = bs.available_players.get_data_frame().copy()
            df_inac  = bs.inactive_players.get_data_frame().copy()
            return df_avail, df_inac
        except Exception:
            time.sleep(sleep * (2 ** attempt))
    return pd.DataFrame(), pd.DataFrame()

def fetch_player_traditional(game_id: str, max_retries=MAX_RETRIES, sleep=SLEEP_BETWEEN_CALLS) -> pd.DataFrame:
    """Player traditional box score (rows: players who recorded a statline)."""
    for attempt in range(max_retries):
        try:
            bs = BoxScoreTraditionalV2(game_id=game_id, timeout=60)
            return bs.player_stats.get_data_frame().copy()
        except Exception:
            time.sleep(sleep * (2 ** attempt))
    return pd.DataFrame()

def derive_home_away(gdf: pd.DataFrame):
    """Given the two team rows for a game from LeagueGameLog, return (home, away, date)."""
    game_date = gdf["GAME_DATE"].iloc[0] if "GAME_DATE" in gdf.columns else None
    home_row = gdf[gdf["MATCHUP"].str.contains("vs.", na=False)]
    away_row = gdf[gdf["MATCHUP"].str.contains("@",   na=False)]
    home = home_row["TEAM_ABBREVIATION"].iloc[0] if not home_row.empty else None
    away = away_row["TEAM_ABBREVIATION"].iloc[0] if not away_row.empty else None
    return home, away, game_date

def write_rows(outfile: Path, rows: list[dict]):
    """Append multiple rows to CSV immediately."""
    if not rows:
        return
    pd.DataFrame(rows).to_csv(outfile, mode="a", index=False, header=False)

# --- Main ---------------------------------------------------------------------
def main():
    # Load universe of players to track
    players_master = load_players_list(PLAYERS_CSV)
    tracked_ids = set(players_master["PERSON_ID"].astype(int).tolist())
    name_lut = dict(zip(players_master["PERSON_ID"].astype(int), players_master["PLAYER_NAME"]))

    ensure_header(OUTFILE)
    done_ids = load_done_ids(OUTFILE)

    tlog = fetch_team_gamelog(SEASON, SEASON_TYPE)
    if tlog.empty:
        raise RuntimeError("LeagueGameLog returned no rows. Check SEASON / SEASON_TYPE.")

    groups = list(tlog.groupby("GAME_ID"))
    groups.sort(key=lambda kv: kv[1]["GAME_DATE"].iloc[0])

    total = len(groups)
    wrote_games = 0
    skipped = 0

    print(f"Found {total} game_ids for {SEASON} ({SEASON_TYPE}). Output: {OUTFILE}")

    for idx, (gid, gdf) in enumerate(groups, start=1):
        gid = str(gid)
        if gid in done_ids:
            skipped += 1
            if idx % 25 == 0 or idx == total:
                print(f"[{idx}/{total}] Skipped already done GAME_ID {gid} (skipped={skipped})")
            continue

        home, away, game_date = derive_home_away(gdf)

        # Pull participation ingredients
        df_avail, df_inac = fetch_boxscore_summary(gid)
        df_play = fetch_player_traditional(gid)
        time.sleep(SLEEP_BETWEEN_CALLS)

        # Build sets
        avail_ids = set(pd.to_numeric(df_avail.get("PERSON_ID", pd.Series(dtype="Int64")), errors="coerce").dropna().astype(int).tolist()) if not df_avail.empty else set()
        inact_ids = set(pd.to_numeric(df_inac.get("PERSON_ID", pd.Series(dtype="Int64")),  errors="coerce").dropna().astype(int).tolist()) if not df_inac.empty else set()
        played_ids = set(pd.to_numeric(df_play.get("PLAYER_ID", pd.Series(dtype="Int64")), errors="coerce").dropna().astype(int).tolist()) if not df_play.empty else set()

        # --- KEY CHANGE: only track players actually on this game's rosters ---
        rostered_ids = (avail_ids | inact_ids | played_ids) & tracked_ids

        # Team abbreviations per rostered player
        team_cols = ["PERSON_ID", "TEAM_ABBREVIATION"]
        team_map_df = pd.concat([
            df_avail[team_cols] if not df_avail.empty and all(c in df_avail.columns for c in team_cols) else pd.DataFrame(columns=team_cols),
            df_inac[team_cols]  if not df_inac.empty  and all(c in df_inac.columns  for c in team_cols) else pd.DataFrame(columns=team_cols),
        ], axis=0, ignore_index=True).dropna(subset=["PERSON_ID"]).drop_duplicates("PERSON_ID")
        team_map = {int(r.PERSON_ID): r.TEAM_ABBREVIATION for _, r in team_map_df.iterrows()}

        # Name lookup fallback from APIs if your CSV is missing some names
        name_map = name_lut.copy()
        for src_df, id_col, name_col in [
            (df_avail, "PERSON_ID", "PLAYER_NAME"),
            (df_inac, "PERSON_ID", "PLAYER_NAME"),
            (df_play.rename(columns={"PLAYER_ID":"PERSON_ID"}), "PERSON_ID", "PLAYER_NAME"),
        ]:
            if not src_df.empty and all(c in src_df.columns for c in [id_col, name_col]):
                for _, r in src_df.iterrows():
                    pid = pd.to_numeric(r[id_col], errors="coerce")
                    if pd.notna(pid):
                        pid = int(pid)
                        if pid in rostered_ids and not name_map.get(pid):
                            name_map[pid] = r.get(name_col, None)

        # Build rows ONLY for rostered tracked players
        rows = []
        for pid in sorted(rostered_ids):
            if pid in inact_ids:
                status = "Inactive"
            elif pid in played_ids:
                status = "Played"
            elif pid in avail_ids:
                status = "DNP"
            else:
                # shouldn't happen because we restricted to rostered_ids, but keep guard
                continue

            rows.append({
                "GAME_ID": gid,
                "GAME_DATE": pd.to_datetime(game_date).date() if pd.notna(game_date) else None,
                "HOME_TEAM": home,
                "AWAY_TEAM": away,
                "TEAM_ABBR": team_map.get(pid),
                "PLAYER_ID": pid,
                "PLAYER_NAME": name_map.get(pid),
                "STATUS": status
            })

        # If neither team had any tracked players, write nothing for this game
        if rows:
            write_rows(OUTFILE, rows)

        done_ids.add(gid)
        wrote_games += 1

        # Live summary (only rostered tracked players)
        played_ct   = sum(1 for r in rows if r["STATUS"] == "Played")
        dnp_ct      = sum(1 for r in rows if r["STATUS"] == "DNP")
        inactive_ct = sum(1 for r in rows if r["STATUS"] == "Inactive")
        print(
            f"[{idx}/{total}] {gid}  {away} @ {home}  date={pd.to_datetime(game_date).date() if pd.notna(game_date) else None}  "
            f"TrackedRostered={len(rows)}  Played={played_ct}  DNP={dnp_ct}  Inactive={inactive_ct}"
        )

    print(f"Done. Processed {wrote_games} games. Skipped {skipped} already present.")

if __name__ == "__main__":
    main()


Found 1230 game_ids for 2024-25 (Regular Season). Output: player_participation_incremental_202425.csv
[4/1230] 0022400064  BKN @ ATL  date=2024-10-23  TrackedRostered=0  Played=0  DNP=0  Inactive=0


KeyboardInterrupt: 

In [3]:
## Pace
# --- Packages -----------------------------------------------------------------
import os
import time
from pathlib import Path
import pandas as pd

from nba_api.stats.endpoints import LeagueGameLog, BoxScoreAdvancedV2

# --- Config -------------------------------------------------------------------
SEASON = "2015-16"             # Changed to 2020-21
SEASON_TYPE = "Regular Season" # "Regular Season", "Playoffs", etc.
SLEEP_BETWEEN_CALLS = 1.2      # seconds between requests
MAX_RETRIES = 4
OUTFILE = Path(f"game_pace_incremental_{SEASON.replace('-', '')}.csv")  # -> game_pace_incremental_202021.csv

# --- Helpers ------------------------------------------------------------------
def load_done_ids(outfile: Path) -> set:
    """Return set of GAME_IDs already written to the CSV."""
    if not outfile.exists():
        return set()
    try:
        done = pd.read_csv(outfile, usecols=["GAME_ID"], dtype=str)
        return set(done["GAME_ID"].astype(str))
    except Exception:
        return set()

def ensure_header(outfile: Path):
    """Create file with header if it doesn't exist."""
    if outfile.exists():
        return
    cols = ["GAME_ID", "GAME_DATE", "HOME_TEAM", "AWAY_TEAM", "PACE_GAME"]
    pd.DataFrame(columns=cols).to_csv(outfile, index=False)

def fetch_team_gamelog(season: str, season_type: str) -> pd.DataFrame:
    gl = LeagueGameLog(
        season=season,
        season_type_all_star=season_type,
        player_or_team_abbreviation="T",
        timeout=60
    )
    df = gl.get_data_frames()[0].copy()
    if "GAME_DATE" in df.columns:
        df["GAME_DATE"] = pd.to_datetime(df["GAME_DATE"])
    return df[["GAME_ID", "GAME_DATE", "TEAM_ABBREVIATION", "MATCHUP"]].copy()

def fetch_team_advanced(game_id: str, max_retries=MAX_RETRIES, sleep=SLEEP_BETWEEN_CALLS) -> pd.DataFrame:
    for attempt in range(max_retries):
        try:
            bs = BoxScoreAdvancedV2(game_id=game_id, timeout=60)
            try:
                team_df = bs.team_stats.get_data_frame().copy()
            except AttributeError:
                team_df = bs.get_data_frames()[1].copy()
            return team_df
        except Exception:
            time.sleep(sleep * (2 ** attempt))
    return pd.DataFrame()

def derive_home_away(gdf: pd.DataFrame):
    """Given the two team rows for a game from LeagueGameLog, return (home, away, date)."""
    game_date = gdf["GAME_DATE"].iloc[0] if "GAME_DATE" in gdf.columns else None
    home_row = gdf[gdf["MATCHUP"].str.contains("vs.", na=False)]
    away_row = gdf[gdf["MATCHUP"].str.contains("@",   na=False)]
    home = home_row["TEAM_ABBREVIATION"].iloc[0] if not home_row.empty else None
    away = away_row["TEAM_ABBREVIATION"].iloc[0] if not away_row.empty else None
    return home, away, game_date

def write_one_row(outfile: Path, row: dict):
    """Append a single row to CSV immediately."""
    pd.DataFrame([row]).to_csv(outfile, mode="a", index=False, header=False)

# --- Main ---------------------------------------------------------------------
def main():
    ensure_header(OUTFILE)
    done_ids = load_done_ids(OUTFILE)

    tlog = fetch_team_gamelog(SEASON, SEASON_TYPE)
    if tlog.empty:
        raise RuntimeError("LeagueGameLog returned no rows. Check SEASON / SEASON_TYPE.")

    groups = list(tlog.groupby("GAME_ID"))
    groups.sort(key=lambda kv: kv[1]["GAME_DATE"].iloc[0])

    total = len(groups)
    wrote = 0
    skipped = 0

    print(f"Found {total} game_ids for {SEASON} ({SEASON_TYPE}). Output: {OUTFILE}")

    for idx, (gid, gdf) in enumerate(groups, start=1):
        gid = str(gid)
        if gid in done_ids:
            skipped += 1
            if idx % 25 == 0 or idx == total:
                print(f"[{idx}/{total}] Skipped already done GAME_ID {gid} (skipped={skipped})")
            continue

        home, away, game_date = derive_home_away(gdf)

        team_adv = fetch_team_advanced(gid)
        time.sleep(SLEEP_BETWEEN_CALLS)

        if team_adv.empty or "PACE" not in team_adv.columns:
            print(f"[{idx}/{total}] GAME_ID {gid}: no advanced data yet. Will try next time.")
            continue

        pace_vals = pd.to_numeric(team_adv["PACE"], errors="coerce").dropna().tolist()
        if len(pace_vals) == 0:
            print(f"[{idx}/{total}] GAME_ID {gid}: PACE all NaN, skipping for now.")
            continue
        avg_pace = sum(pace_vals) / len(pace_vals)

        row = {
            "GAME_ID": gid,
            "GAME_DATE": pd.to_datetime(game_date).date() if pd.notna(game_date) else None,
            "HOME_TEAM": home,
            "AWAY_TEAM": away,
            "PACE_GAME": round(avg_pace, 3),
        }

        write_one_row(OUTFILE, row)
        done_ids.add(gid)
        wrote += 1

        print(f"[{idx}/{total}] Wrote {gid}  {away} @ {home}  date={row['GAME_DATE']}  PACE={row['PACE_GAME']}")

    print(f"Done. Wrote {wrote} new games. Skipped {skipped} already present.")

if __name__ == "__main__":
    main()


Found 1230 game_ids for 2015-16 (Regular Season). Output: game_pace_incremental_201516.csv
[1/1230] Wrote 0021500001  DET @ ATL  date=2015-10-27  PACE=98.0
[2/1230] Wrote 0021500002  CLE @ CHI  date=2015-10-27  PACE=98.0
[3/1230] Wrote 0021500003  NOP @ GSW  date=2015-10-27  PACE=102.5
[4/1230] Wrote 0021500004  WAS @ ORL  date=2015-10-28  PACE=99.0
[5/1230] Wrote 0021500005  PHI @ BOS  date=2015-10-28  PACE=102.5


KeyboardInterrupt: 

In [5]:
## Players
# players_2024_25.py
# -------------------------------------------
# Get all players who appeared in the 2024-25 Regular Season
# and save PERSON_ID + PLAYER_NAME to a CSV.

import time
from pathlib import Path

import pandas as pd
from nba_api.stats.endpoints import LeagueGameLog

SEASON = "2023-24"
SEASON_TYPE = "Regular Season"   # change to "Playoffs" if needed
OUTFILE = Path("players_master_2024_25.csv")
TIMEOUT = 60
SLEEP = 1.0
MAX_RETRIES = 4

def fetch_league_gamelog_players(season: str, season_type: str) -> pd.DataFrame:
    """Return unique players (PERSON_ID, PLAYER_NAME) who appeared this season."""
    for attempt in range(MAX_RETRIES):
        try:
            gl = LeagueGameLog(
                season=season,
                season_type_all_star=season_type,
                player_or_team_abbreviation="P",
                timeout=TIMEOUT
            )
            df = gl.league_game_log.get_data_frame().copy()
            if df.empty:
                return pd.DataFrame(columns=["PERSON_ID", "PLAYER_NAME"])
            out = (df[["PLAYER_ID", "PLAYER_NAME"]]
                   .drop_duplicates()
                   .rename(columns={"PLAYER_ID": "PERSON_ID"}))
            # ensure types and a nice sort
            out["PERSON_ID"] = pd.to_numeric(out["PERSON_ID"], errors="coerce").astype("Int64")
            out = out.dropna(subset=["PERSON_ID"]).astype({"PERSON_ID": int})
            out = out.sort_values(["PLAYER_NAME", "PERSON_ID"]).reset_index(drop=True)
            return out[["PERSON_ID", "PLAYER_NAME"]]
        except Exception as e:
            if attempt == MAX_RETRIES - 1:
                raise
            time.sleep(SLEEP * (2 ** attempt))

def main():
    players = fetch_league_gamelog_players(SEASON, SEASON_TYPE)
    if players.empty:
        raise RuntimeError(f"No players found for {SEASON} ({SEASON_TYPE}). "
                           "If the season hasn't started, try again later or check your network.")
    players.to_csv(OUTFILE, index=False)
    print(f"Saved {len(players)} players to {OUTFILE.resolve()}")

if __name__ == "__main__":
    main()


Saved 569 players to /Users/jacksonmunro/Desktop/QSS_82/players_master_2024_25.csv
